In [1]:
import asyncio
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_openai import ChatOpenAI
from pprint import pprint

In [3]:
# Define the model
model = ChatOpenAI(
    model="gpt-4.1-mini",
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)

# Time Conversion

In [4]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

UnsupportedOperation: fileno

In [5]:
import sys
import os

# Create a function for the time conversion MCP
async def time_conversion_mcp():
    # Workaround for Jupyter stderr issue on Windows
    original_stderr = sys.stderr
    stderr_file = open('nul', 'w') if sys.platform == 'win32' else open('/dev/null', 'w')
    sys.stderr = stderr_file
    
    try:
        client = MultiServerMCPClient(
            {
                "time": {
                    "transport": "stdio",
                    "command": "uvx",
                    "args": [
                        "mcp-server-time",
                        "--local-timezone=America/New_York"
                    ]
                }
            }
        )
        tools = await client.get_tools()
        
        # Restore stderr before using the agent
        sys.stderr = original_stderr
        stderr_file.close()

        agent = create_agent(
            model=model, 
            tools=tools,
        )
        question = HumanMessage(content="What time is it?")
        response = await agent.ainvoke(
            {"messages": [question]}
        )
        pprint(response)
    finally:
        # Make sure stderr is restored even if there's an error
        if sys.stderr != original_stderr:
            sys.stderr = original_stderr
            stderr_file.close()

# In Jupyter, just await it directly
await time_conversion_mcp()

UnsupportedOperation: fileno

## Why MCP with stdio Transport Doesn't Work in Jupyter on Windows

### The Problem
MCP servers using `stdio` transport fail in Jupyter notebooks on Windows with `UnsupportedOperation: fileno` error.

### Root Cause
1. **Jupyter's stderr limitation**: Jupyter notebooks use a custom `OutStream` class for stderr that doesn't support the `fileno()` method
2. **Windows subprocess requirement**: On Windows, `subprocess.Popen()` needs a real file descriptor when redirecting stderr
3. **MCP's internal behavior**: The MCP library internally captures and uses `sys.stderr` when creating subprocesses, and you can't control this from outside

### The Error Chain
MCP Client -> create subprocess with uvx
-> needs to redirect stderr for logging
-> calls subprocess.Popen(stderr=sys.stderr)
-> Windows tries to get file descriptor: stderr.fileno()
-> Jupyter's OutStream raises UnsupportedOperation


### Why Simple Workarounds Fail
- **Redirecting `sys.stderr` before import**: MCP captures stderr at subprocess creation time, not import time
- **Temporary file approach**: Same issue - MCP still uses the original Jupyter stderr internally
- **Context managers**: Can't intercept the stderr parameter deep inside MCP's subprocess creation

### Solutions
1. ✅ **Use regular Python scripts** - Works perfectly, no limitations
2. ⚠️ **Monkey-patch `subprocess.Popen`** - Hacky but might work in Jupyter
3. ✅ **Run script from Jupyter** - Use `subprocess.run()` to call a separate Python script
4. ❌ **Use different MCP transport** - If available (not stdio)

### Bottom Line
This is a fundamental incompatibility: **Windows + Jupyter + MCP stdio transport = broken**

The issue exists because:
- Unix-like systems can handle file-like objects more flexibly
- Windows requires actual OS-level file handles for subprocess I/O redirection
- Jupyter's virtual stderr doesn't provide these handles

# Travel Planner

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
    config
    )

In [ ]:
from pprint import pprint

pprint(response)

In [ ]:
print(response["messages"][-1].content)